In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 1.3 The Double Pendulum

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume I — Elementary Mechanics",
    number="1.3",
    title="The Double Pendulum",
    blurb="From a two-line Lagrangian to deterministic chaos: equations of "
    "motion, energy conservation, the phase portrait, sensitive dependence, "
    "and a kernel-free animation.",
    difficulty="intermediate",
    estimate="60–90 min",
)

## Notebook overview

The double pendulum (a pendulum hung from the end of another pendulum) is
the smallest mechanical system most students meet that is genuinely
*chaotic*. It is fully deterministic: given the initial angles and angular
velocities, Newton (or, more comfortably, Lagrange) fixes the entire future.
Yet two starts that differ by a hair diverge exponentially, so in practice
the motion is unpredictable. That tension (determinism without
predictability) is the whole pedagogical point, and it is best *seen* rather
than described, which is exactly what a notebook is good for.

We will (1) set up coordinates and the Lagrangian, (2) obtain the equations
of motion (once by hand, once symbolically with SymPy as a check), (3)
integrate them numerically, (4) verify the solution by watching the total
energy stay constant, (5) draw the phase portrait, (6) measure sensitive
dependence on initial conditions, and (7) animate the motion.

> **Scope.** This is a working review, not a textbook chapter. For the
> analytical mechanics behind it, see Nolting, *Theoretische Physik 1–2*
> (Klassische Mechanik) {cite}`nolting1,nolting2`; Goldstein, Poole & Safko,
> *Classical Mechanics* {cite}`goldstein`; and, for the geometry of chaos,
> Strogatz, *Nonlinear Dynamics and Chaos* {cite}`strogatz`.

## Theory in brief

### Coordinates and the Lagrangian

Take two point masses $m_1, m_2$ on massless rigid rods of lengths
$\ell_1, \ell_2$. Measure both angles $\theta_1, \theta_2$ from the downward
vertical. The bob positions are

$$
x_1 = \ell_1 \sin\theta_1, \quad y_1 = -\ell_1 \cos\theta_1, \qquad
x_2 = x_1 + \ell_2 \sin\theta_2, \quad y_2 = y_1 - \ell_2 \cos\theta_2 .
$$

Differentiating gives the kinetic energy $T = \tfrac12 m_1(\dot x_1^2 + \dot
y_1^2) + \tfrac12 m_2(\dot x_2^2 + \dot y_2^2)$ and the potential energy
$V = m_1 g y_1 + m_2 g y_2$. The Lagrangian is $\mathcal{L} = T - V$:

$$
\mathcal{L} = \tfrac12 (m_1+m_2)\ell_1^2\dot\theta_1^2
  + \tfrac12 m_2 \ell_2^2 \dot\theta_2^2
  + m_2 \ell_1 \ell_2 \dot\theta_1 \dot\theta_2 \cos(\theta_1-\theta_2)
  + (m_1+m_2) g \ell_1 \cos\theta_1 + m_2 g \ell_2 \cos\theta_2 .
$$

### Equations of motion

The Euler–Lagrange equations $\frac{d}{dt}\frac{\partial \mathcal L}{\partial
\dot\theta_i} - \frac{\partial \mathcal L}{\partial \theta_i} = 0$ yield two
coupled second-order ODEs. Solved for the angular accelerations (writing
$\Delta = \theta_1-\theta_2$ and $D = 2m_1 + m_2 - m_2\cos 2\Delta$):

$$
\ddot\theta_1 = \frac{-g(2m_1+m_2)\sin\theta_1 - m_2 g \sin(\theta_1-2\theta_2)
  - 2\sin\Delta\, m_2(\dot\theta_2^2 \ell_2 + \dot\theta_1^2 \ell_1\cos\Delta)}
  {\ell_1 D},
$$

$$
\ddot\theta_2 = \frac{2\sin\Delta\,\big(\dot\theta_1^2 \ell_1 (m_1+m_2)
  + g(m_1+m_2)\cos\theta_1 + \dot\theta_2^2 \ell_2 m_2 \cos\Delta\big)}
  {\ell_2 D}.
$$

We will reproduce these symbolically below rather than trust the algebra.

### Small oscillations

For $|\theta_i| \ll 1$ the system linearises to two coupled harmonic
oscillators with two **normal modes**. For the equal case
$m_1=m_2=m,\ \ell_1=\ell_2=\ell$ the mode frequencies are

$$
\omega_\pm^2 = (2 \pm \sqrt{2})\,\frac{g}{\ell}.
$$

(Taylor, *Classical Mechanics*, Ch. 11, carries the linearisation out in full
for exactly this system; the general eigenvalue machinery behind it is the
subject of [§2.7](../02-classical-mechanics/small-oscillations.ipynb).)

This is our first independent check: any correct integrator, started at tiny
amplitude, must oscillate at these frequencies.

### The geometry

Two rods hang from a fixed pivot: the first, of length $\ell_1$, carries bob
$m_1$ at angle $\theta_1$ from the downward vertical; the second, length
$\ell_2$, hangs from $m_1$ and carries $m_2$ at its own angle $\theta_2$. Both
angles are measured from vertical, so the configuration is the pair
$(\theta_1, \theta_2)$. The animation later sets these in motion; the still
schematic fixes what the symbols mean.

In [ ]:
# (solution hidden on the public site)


---
## Setup

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

from ecp import validate
from ecp.animate import show

rng = np.random.default_rng(0)

# Physical parameters (SI). Equal masses and lengths unless stated otherwise.
G = 9.81
M1 = M2 = 1.0
L1 = L2 = 1.0

## Exercise 1 — Implement the equations of motion

Write the right-hand side for the state vector
$\mathbf{y} = (\theta_1, \dot\theta_1, \theta_2, \dot\theta_2)$.

1. Implement `deriv(t, y)` returning
   $(\dot\theta_1, \ddot\theta_1, \dot\theta_2, \ddot\theta_2)$ from the
   acceleration expressions above (note the shared denominator
   $D = 2m_1 + m_2 - m_2\cos 2\Delta$).
2. Cross-check it against an independent SymPy derivation from the Lagrangian
   (the validation below does exactly this at a random state).

In [ ]:
# (solution hidden on the public site)


### Validation 1 — symbolic cross-check of the equations of motion

Rather than trust the hand algebra, derive the accelerations from the
Lagrangian with SymPy and confirm they agree with `deriv` at a random state.

In [ ]:
import sympy as sp

t = sp.symbols("t")
m1, m2, l1, l2, g = sp.symbols("m1 m2 l1 l2 g", positive=True)
th1 = sp.Function("th1")(t)
th2 = sp.Function("th2")(t)

x1, y1 = l1 * sp.sin(th1), -l1 * sp.cos(th1)
x2, y2 = x1 + l2 * sp.sin(th2), y1 - l2 * sp.cos(th2)
T = sp.Rational(1, 2) * m1 * (sp.diff(x1, t) ** 2 + sp.diff(y1, t) ** 2) + sp.Rational(
    1, 2
) * m2 * (sp.diff(x2, t) ** 2 + sp.diff(y2, t) ** 2)
V = m1 * g * y1 + m2 * g * y2
Lag = sp.simplify(T - V)

# Euler–Lagrange -> solve for the two angular accelerations.
EL = [sp.diff(sp.diff(Lag, sp.diff(q, t)), t) - sp.diff(Lag, q) for q in (th1, th2)]
acc = sp.solve(EL, [sp.diff(th1, t, 2), sp.diff(th2, t, 2)], dict=True)[0]
dd1 = sp.lambdify(
    (th1, sp.diff(th1, t), th2, sp.diff(th2, t), m1, m2, l1, l2, g),
    acc[sp.diff(th1, t, 2)],
)
dd2 = sp.lambdify(
    (th1, sp.diff(th1, t), th2, sp.diff(th2, t), m1, m2, l1, l2, g),
    acc[sp.diff(th2, t, 2)],
)

probe = [0.7, -0.4, 1.9, 0.25]
hand = deriv(0.0, probe)
symb = [
    probe[1],
    dd1(probe[0], probe[1], probe[2], probe[3], M1, M2, L1, L2, G),
    probe[3],
    dd2(probe[0], probe[1], probe[2], probe[3], M1, M2, L1, L2, G),
]
validate.close(hand, symb, "hand-derived EOM match the SymPy derivation", rtol=1e-9)

## Exercise 2 — Integrate and view the motion

1. Integrate from a large-amplitude start with `scipy.integrate.solve_ivp`
   (`DOP853` at tight tolerances — chaotic systems are unforgiving of sloppy
   integration).
2. Plot both angles against time.

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Total energy as a validation of the integrator

A correct, well-integrated trajectory conserves total mechanical energy — the
canonical "did I integrate it right?" check.

1. Compute $E(t) = T + V$ along the solution (the `total_energy` helper).
2. Confirm the relative drift is tiny.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(E, "total energy conserved along the trajectory", rel_drift=1e-4)

## Exercise 4 — Small-amplitude normal modes

At tiny amplitude the system linearises, and its oscillation must sit at a
normal-mode frequency $\omega_\pm^2 = (2\pm\sqrt2)\,g/\ell$ — a check of the
*physics* against linear theory.

1. Start at small, in-phase angles (which excites mainly the low mode) and
   integrate a long run.
2. Take the power spectrum of $\theta_1(t)$ (`numpy.fft.rfft`/`rfftfreq`) and
   locate the dominant peak.
3. Confirm it matches $\omega_-=\sqrt{(2-\sqrt2)\,g/\ell}$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    omega_measured,
    omega_minus,
    "small-amplitude peak matches the low normal mode",
    rtol=3e-2,
)  # tolerance covers the discrete FFT bin width

## Exercise 5 — Phase portrait

1. Integrate a long, large-amplitude run and plot the trajectory in the
   $(\theta_2, \dot\theta_2)$ plane.
2. Note the signature: the curve never closes and fills a region — non-integrable
   motion, in contrast to the closed loops of a single pendulum.

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Sensitive dependence on initial conditions

1. Run two trajectories whose initial angles differ by $\delta_0 = 10^{-6}$ rad
   and track their separation $\delta(t)$ in configuration space.
2. Early on it grows roughly as $\delta(t)\sim\delta_0 e^{\lambda t}$: estimate
   the largest Lyapunov exponent $\lambda$ from the slope of $\ln\delta$
   (`numpy.polyfit` over the growth window, before saturation).
3. Confirm $\lambda$ is **positive**: the formal definition of chaos.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.positive(slope, "largest Lyapunov exponent is positive (chaotic)")

## Exercise 7 — Animate the motion

1. Build a `FuncAnimation` of the two rods and the lower-bob trail.
2. Render it with `ecp.animate.show`, which pre-renders to a self-contained
   HTML5 player so it plays on the static course website with no Python kernel.

In [ ]:
# (solution hidden on the public site)


## Notebook summary

- The double pendulum's equations of motion (matched against a SymPy derivation) integrated
  with total energy conserved as the integrator check; the small-amplitude normal modes and
  the phase portrait.
- Sensitive dependence on initial conditions (a positive largest Lyapunov exponent), the
  signature of deterministic chaos, with the motion animated.

## Outlook

- Replace the point bobs with physical rods (moments of inertia) and rederive
  the Lagrangian. How do the normal-mode frequencies change?
- Build a **Poincaré section**: record $(\theta_2,\dot\theta_2)$ each time
  $\theta_1=0$ with $\dot\theta_1>0$, and watch regular islands dissolve into
  a chaotic sea as the energy increases.
- Estimate the Lyapunov exponent properly with the Benettin renormalisation
  algorithm instead of a single-shot separation fit.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()